In [ ]:
import numpy as np
import matplotlib.pyplot as plt
#from numba import njit

lattice_size = 100
T = 2 # K
kB = 1 # J/K


beta = 1/(kB*T)
lattice = np.random.random((lattice_size, lattice_size))
lattice = np.where(lattice>0.5, 1, -1).astype(int)


def getEnergy(lattice, J = 1):
    LatShiftedUp = np.roll(lattice, shift=-1, axis=0) # roll ya hace wrap
    LatShiftedRight = np.roll(lattice, shift=1, axis=1)

    UpEnergyMat = lattice*LatShiftedUp
    RightEnergyMat = lattice*LatShiftedRight

    Einit = -1*J*(np.sum(UpEnergyMat)+np.sum(RightEnergyMat))

    return Einit

def FlipRandSpin(lattice, Einit):
    LatLen = lattice.shape[0]
    RandIdx_i, RandIdx_j = np.random.uniform(0, LatLen, 2).astype(int)

    # si está en el borde hay que tener en cuenta frontera periódica
    #print(lattice.shape)
    PaddedLat = np.pad(lattice, pad_width=((1, 1), (1, 1)), mode = 'wrap')
    RandIdx_i, RandIdx_j = RandIdx_i + 1, RandIdx_j + 1

    def LocalE(lattice, idx_i, idx_j):
        MiniLat = lattice[idx_i-1:idx_i+2, idx_j-1:idx_j+2].copy()# matriz 3x3
        MiniLat[0][0] = MiniLat[0][-1] = MiniLat[-1][0] = MiniLat[-1][-1] = 0 # anulamos esquinas
        MiniLat = np.pad(MiniLat, pad_width=((1, 1), (1, 1)), mode = 'constant', constant_values=0) # pad de ceros para no contar energía extra
        
        return getEnergy(MiniLat) # energia de la region local del spin seleccionado
    
    EinitLocal = LocalE(PaddedLat, RandIdx_i, RandIdx_j) # energia sin flip

    PaddedLat[RandIdx_i][RandIdx_j] = PaddedLat[RandIdx_i][RandIdx_j]*-1 #flipeamos el spin

    EendLocal = LocalE(PaddedLat, RandIdx_i, RandIdx_j) # energia con flip

    deltaE = EendLocal-EinitLocal


    
    if deltaE>0:
        p = np.exp(-beta*deltaE)
        r = np.random.uniform(0, 1, size=1)[0]
        
        if r<=p:
            ReturnLattice = PaddedLat[1:-1, 1:-1]# quitar padding
            ReturnEnergy = Einit+deltaE

        else:
            PaddedLat[RandIdx_i][RandIdx_j] = PaddedLat[RandIdx_i][RandIdx_j]*-1 #dejamos como estaba
            ReturnLattice = PaddedLat[1:-1, 1:-1] # quitar padding
            ReturnEnergy = Einit

    else:
        ReturnLattice = PaddedLat[1:-1, 1:-1]# quitar padding
        ReturnEnergy = Einit+deltaE
    
    return ReturnLattice, ReturnEnergy

steps = 100
Einit = getEnergy(lattice)

iter_list = []
lattice_list = []
Einit_list = []

for i in range(0, steps, 1):
    lattice, Einit = FlipRandSpin(lattice, Einit)
    lattice_list.append(lattice)
    Einit_list.append(Einit)
    iter_list.append(i)
    if i % 10000 == 0:
        print(f'Iteration {i}/{steps}', flush=True)

magnetization = [np.sum(x)/lattice_size**2 for x in lattice_list]


In [ ]:
fig, (axE, axM) = plt.subplots(nrows=2, ncols=1, figsize=(8, 8),
                               sharex=True,  # compartir eje X si quieres
                               constrained_layout=True)

axE.plot(iter_list, Einit_list, marker='o', color='tab:blue', markersize=3)
axE.set_ylabel('Energía inicial')
axE.set_title('Energía vs Iteraciones')
axE.grid(True)

axM.plot(iter_list, magnetization, marker='o', color='tab:red', markersize=3)
axM.set_xlabel('Iteraciones')
axM.set_ylabel('Magnetización')
axM.set_title('Magnetización vs Iteraciones')
axM.grid(True)

plt.show()